# Text Clustering and Semantic Retrieval

In this notebook, we will study clustering and retrieval techniques.

## Setup

In [ ]:
!pip install nltk torch sentence-transformers faiss-cpu bertopic rank_bm25 datasets umap-learn hdbscan matplotlib plotly nbformat

## Load Dataset

We use the AG News dataset (Zhang et al., 2015): 120 000 news articles across
four categories: *World*, *Sports*, *Business*, and *Sci/Tech*.

For this session we work with a random sample of 5 000 articles to keep
runtimes manageable without a GPU.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

np.random.seed(42)

dataset = load_dataset("ag_news", split="train")
N = 20000
df = dataset.to_pandas().sample(N, random_state=42).reset_index(drop=True)

# The dataset has two columns: 'text' and 'label' (0–3)
label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
df["category"] = df["label"].map(label_names)

print(f"Corpus size : {len(df)} articles")
print(f"\nCategory distribution:")
print(df["category"].value_counts())
df.head(3)

## Text Clustering

### TF-IDF + K-Means

The classic approach: represent each document as a TF-IDF vector, then
cluster with k-means. We know the true number of topics (4), so we set
`k = 4`. In a real scenario we would try with several values of `k`
and inspect the silhouette score.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import nltk
from nltk.corpus import stopwords

# Define list of stop words
nltk.download('stopwords')
stop_words_en = stopwords.words('english')

# TF-IDF representation
vectorizer = TfidfVectorizer(max_features=20000, sublinear_tf=False, norm='l2', 
                             ngram_range=(1, 1), min_df=10, max_df=0.95, stop_words=stop_words_en)
X_tfidf = vectorizer.fit_transform(df["text"])

print(f"TF-IDF matrix shape: {X_tfidf.shape}")

In [ ]:
# K-Means clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
#kmeans = MiniBatchKMeans(n_clusters=4, random_state=42, n_init=20)
df["km_label"] = kmeans.fit_predict(X_tfidf)

# Evaluation
sil = silhouette_score(X_tfidf, df["km_label"], metric="cosine", sample_size=2000)
ari = adjusted_rand_score(df["label"], df["km_label"])

print(f"Silhouette score (cosine) : {sil:.4f}  [range -1 to 1, higher is better]")
print(f"Adjusted Rand Index       : {ari:.4f}  [1.0 = perfect match with true labels]")

In [ ]:
# Run diagnostics
unique, counts = np.unique(df["km_label"], return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"Cluster {cls}: {cnt} docs ({cnt/len(df["km_label"])*100:.1f}%)")

In [ ]:
# Reduce data dimensionality
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Reduce dimensions first
lsa = make_pipeline(
    TruncatedSVD(n_components=10, random_state=42),
    Normalizer()
)
X_lsa = lsa.fit_transform(X_tfidf)

In [ ]:
# Now cluster in dense space
kmeans_dense = KMeans(n_clusters=4, random_state=42, n_init=20)
df["km_dense_label"] = kmeans_dense.fit_predict(X_lsa)

# Evaluation
sil = silhouette_score(X_tfidf, df["km_dense_label"], metric="cosine", sample_size=2000)
ari = adjusted_rand_score(df["label"], df["km_dense_label"])

print(f"Silhouette score (cosine) : {sil:.4f}  [range -1 to 1, higher is better]")
print(f"Adjusted Rand Index       : {ari:.4f}  [1.0 = perfect match with true labels]")

In [ ]:
# Check cluster sizes
unique, counts = np.unique(df["km_dense_label"], return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"Cluster {cls}: {cnt} docs ({cnt/len(df["km_dense_label"])*100:.1f}%)")

#### Inspect cluster top terms

Since the dimensionality of the feature vectors was reduced, it is necessary to project the centroids of each k-means back to word space. This operation returns an approximation of the true centroids that is often useful to read the top terms.

In [ ]:
import numpy as np

# kmeans_dense.cluster_centers_ shape: (n_clusters, n_components)  e.g. (4, 10)
# svd.components_ shape: (n_components, n_features)       e.g. (10, vocab_size)

svd = lsa.named_steps['truncatedsvd']
terms = vectorizer.get_feature_names_out()

# Project centroids back to word space
centroids_word_space = np.dot(kmeans_dense.cluster_centers_, svd.components_)
# shape: (4, vocab_size)

# Top words per cluster
for i in range(4):
    top_indices = centroids_word_space[i].argsort()[::-1][:10]
    top_words = [terms[idx] for idx in top_indices]
    print(f"Cluster {i}: {top_words}")

### BERTopic

BERTopic uses (i) contextual sentence embeddings, (ii) UMAP dimensionality reduction, (iii) HDBSCAN clustering and (iv) c-TF-IDF topic labels. It does not require specifying `k` in advance and can mark noisy documents as `topic −1`.

In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# Step 1: Encode documents
#   We use a lightweight model to keep runtimes short.
#   For higher quality, try "all-mpnet-base-v2".
print("Encoding documents (this may take a few minutes on CPU)...")
#model_name = "all-MiniLM-L6-v2"
model_name = "all-mpnet-base-v2"
encoder = SentenceTransformer(model_name)
embeddings = encoder.encode(df["text"].tolist(), batch_size=64,
                             show_progress_bar=True)
print(f"Embedding matrix shape: {embeddings.shape}")

In [ ]:
# Step 2: Configure sub-models
umap_model  = UMAP(n_components=5, n_neighbors=15, min_dist=0.0,
                   metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=30, metric="euclidean",
                         prediction_data=True)

In [ ]:
# Step 3: Fit BERTopic
topic_model = BERTopic(
    embedding_model=encoder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="english",
    calculate_probabilities=False,
    verbose=False
)

topics, _ = topic_model.fit_transform(df["text"].tolist(), embeddings)
df["bertopic_label"] = topics

n_topics = len(set(topics)) - (1 if -1 in topics else 0)
n_outliers = (df["bertopic_label"] == -1).sum()
print(f"Topics found : {n_topics}")
print(f"Outliers (topic −1): {n_outliers} ({n_outliers/len(df)*100:.1f}%)")

In [ ]:
# Show top words per topic
topic_info = topic_model.get_topic_info()
print(topic_info[topic_info["Topic"] != -1][["Topic", "Count", "Name"]].head(8).to_string(index=False))

In [ ]:
# Interactive visualisation (works in Jupyter / Colab)
fig = topic_model.visualize_topics()
fig.show()

**Question:** Compare the BERTopic topics with the k-means clusters.
Which approach produces more interpretable topics?

## Semantic Retrieval

We now build three retrieval systems on the same corpus and compare them:

| System | Method |
|--------|--------|
| BM25 | Lexical (bag-of-words) |
| Dense | Bi-encoder + FAISS |
| Hybrid | BM25 + Dense (RRF) |

To evaluate them we need queries with known relevant documents.
We create a small hand-made evaluation set with 5 queries per category.

### Evaluation Set

In [ ]:
queries = [
    {"category": "World",    "text": "military conflict troops deployment"},
    {"category": "World",    "text": "diplomatic negotiations peace agreement"},
    {"category": "World",    "text": "United Nations resolution foreign policy"},
    {"category": "World",    "text": "election results government coalition"},
    {"category": "World",    "text": "humanitarian crisis refugees aid"},
    {"category": "Sports",   "text": "championship football tournament results"},
    {"category": "Sports",   "text": "olympic athlete gold medal record"},
    {"category": "Sports",   "text": "basketball playoffs NBA scoring"},
    {"category": "Sports",   "text": "tennis grand slam match victory"},
    {"category": "Sports",   "text": "baseball pitcher strikeouts season"},
    {"category": "Business", "text": "quarterly earnings revenue profit"},
    {"category": "Business", "text": "stock market shares investors trading"},
    {"category": "Business", "text": "central bank interest rates inflation"},
    {"category": "Business", "text": "merger acquisition corporate deal"},
    {"category": "Business", "text": "oil prices energy commodity market"},
    {"category": "Sci/Tech", "text": "artificial intelligence machine learning software"},
    {"category": "Sci/Tech", "text": "space satellite launch NASA mission"},
    {"category": "Sci/Tech", "text": "smartphone processor chip semiconductor"},
    {"category": "Sci/Tech", "text": "cybersecurity data breach vulnerability"},
    {"category": "Sci/Tech", "text": "climate research scientific study findings"},
]

query_df  = pd.DataFrame(queries)
corpus_df = df.reset_index(drop=True)

print(f"Corpus : {len(corpus_df)} documents")
print(f"Queries: {len(query_df)}")
print(query_df.head(4).to_string(index=False))

In [ ]:
def recall_at_k(retrieved_ids, relevant_ids, k=10):
    """Fraction of relevant docs found in the top-k results."""
    top_k = set(retrieved_ids[:k])
    return len(top_k & relevant_ids) / len(relevant_ids)

def precision_at_k(retrieved_docs, relevant_docs, k):
    """
    retrieved_docs: ordered list of doc IDs returned by the system
    relevant_docs: set of doc IDs that are truly relevant
    k: cutoff
    """
    retrieved_at_k = set(retrieved_docs[:k])
    relevant = set(relevant_docs)
    
    hits = retrieved_at_k & relevant
    
    return len(hits) / k

def reciprocal_rank(retrieved_ids, relevant_ids):
    """Rank of the first relevant document (1/rank)."""
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0

def evaluate(results_per_query, relevant_per_query, k=10):
    """Compute mean Recall@k and MRR over all queries."""
    precisions, rrs = [], []
    for q_id, retrieved in results_per_query.items():
        relevant = relevant_per_query[q_id]
        precisions.append(precision_at_k(retrieved, relevant, k))
        rrs.append(reciprocal_rank(retrieved, relevant))
    return {
        f"Precision@{k}": np.mean(precisions),
        "MRR"        : np.mean(rrs)
    }

# Build relevance sets: for each query, relevant = same category in corpus
relevant_per_query = {}
for q_id, row in query_df.iterrows():
    relevant_ids = set(corpus_df.index[corpus_df["category"] == row["category"]])
    relevant_per_query[q_id] = relevant_ids

print(f"Example: query 0 has {len(relevant_per_query[0])} relevant documents")

### BM25 Lexical Retrieval

`rank_bm25` provides a pure-Python BM25 implementation.
We tokenize by whitespace and lowercase (simple but effective for English news).


In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize corpus
tokenized_corpus = [doc.lower().split() for doc in corpus_df["text"]]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built")

In [ ]:
def bm25_search(query_text, top_k=50):
    tokens = query_text.lower().split()
    scores = bm25.get_scores(tokens)
    top_ids = np.argsort(scores)[::-1][:top_k]
    return list(top_ids)

# Retrieve for all queries
bm25_results = {}
for q_id, row in query_df.iterrows():
    bm25_results[q_id] = bm25_search(row["text"], top_k=50)
print(bm25_results[1])

In [ ]:
metrics_bm25 = evaluate(bm25_results, relevant_per_query, k=10)
print("BM25 results:")
for k, v in metrics_bm25.items():
    print(f"  {k}: {v:.4f}")

### Dense Retrieval with FAISS

We regenerate the same embeddings computed for BERTopic. No extra encoding needed.
We build an `IndexHNSW` index, which offers an excellent speed/recall trade-off
without requiring a training step.

As a model for dense retrieval, we will use `msmarco-distilbert-base-v4`, a lightweight bi-encoder based on DistilBERT, fine-tuned on the MS MARCO passage retrieval dataset to match short natural language queries against longer documents using dot-product similarity.

In [ ]:
from sentence_transformers import SentenceTransformer

# Step 1: Encode documents
print("Encoding documents (this may take a few minutes on CPU)...")
model_name = "msmarco-distilbert-base-v4"
encoder = SentenceTransformer(model_name)
embeddings = encoder.encode(df["text"].tolist(), batch_size=64,
                             show_progress_bar=True)
print(f"Embedding matrix shape: {embeddings.shape}")

In [ ]:
import faiss

# Embeddings for the whole corpus
corpus_embeddings = np.array([embeddings[i] for i in corpus_df.index]).astype("float32")

# Normalize for cosine similarity (inner product on unit vectors = cosine)
faiss.normalize_L2(corpus_embeddings)

d = corpus_embeddings.shape[1]         # embedding dimension

# Build IndexHNSWFlat
index = faiss.IndexHNSWFlat(d, 32)     # 32 neighbours per graph node
index.hnsw.efConstruction = 200        # higher = better index quality

# Build IndexFlatIP
# index = faiss.IndexFlatIP(d)

index.add(corpus_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dimension {d}")

In [ ]:
query_embeddings = encoder.encode(query_df["text"].tolist(),
                                  normalize_embeddings=True).astype("float32")

K = 50
_, I = index.search(query_embeddings, K)

dense_results = {}
for q_idx, q_id in enumerate(query_df.index):
    dense_results[q_id] = list(I[q_idx])

metrics_dense = evaluate(dense_results, relevant_per_query, k=10)
print("Dense retrieval results:")
for k, v in metrics_dense.items():
    print(f"  {k}: {v:.4f}")

### Hybrid Search with Reciprocal Rank Fusion

RRF merges the ranked lists from BM25 and dense retrieval using only rank
positions — no score normalisation required.

$$\text{RRF}(d) = \sum_{r} \frac{1}{k + r_r(d)}, \quad k = 60$$


In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Merge multiple ranked lists using RRF.
    ranked_lists: list of lists of doc IDs (ordered best → worst)
    Returns: list of doc IDs sorted by descending RRF score
    """
    scores = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

hybrid_results = {}
for q_id in query_df.index:
    fused = reciprocal_rank_fusion([bm25_results[q_id], dense_results[q_id]])
    hybrid_results[q_id] = fused

metrics_hybrid = evaluate(hybrid_results, relevant_per_query, k=10)
print("Hybrid (RRF) results:")
for k, v in metrics_hybrid.items():
    print(f"  {k}: {v:.4f}")

### Comparison

In [ ]:
import matplotlib.pyplot as plt

results_df = pd.DataFrame(
    [metrics_bm25, metrics_dense, metrics_hybrid],
    index=["BM25", "Dense (FAISS)", "Hybrid (RRF)"]
)
print(results_df.round(4).to_string())

# Bar chart
ax = results_df.plot(kind="bar", figsize=(8, 4), rot=0, colormap="Set2",
                     edgecolor="black", linewidth=0.5)
ax.set_title("Retrieval performance comparison", fontsize=13)
ax.set_ylabel("Score")
ax.legend(loc="lower right")
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3, fontsize=9)
plt.tight_layout()
plt.show()

**Question:** Which system performs best on this corpus?
Does hybrid search always outperform both individual systems?
Under what conditions might BM25 beat the dense retriever?


## System Comparison

Type any query in natural language. The cell below runs all three retrieval
systems and shows the top-5 results side by side.


In [ ]:
def search_all(query, top_k=5):
    # BM25
    bm25_ids = bm25_search(query, top_k=top_k)

    # Dense
    q_emb = encoder.encode([query], normalize_embeddings=True).astype("float32")
    _, I_q = index.search(q_emb, top_k)
    dense_ids = list(I_q[0])

    # Hybrid
    hybrid_ids = reciprocal_rank_fusion([bm25_ids, dense_ids])[:top_k]

    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print(f"{'='*70}\n")
    for label, ids in [("BM25", bm25_ids), ("Dense", dense_ids), ("Hybrid", hybrid_ids)]:
        print(f"== {label} ======================================")
        for rank, doc_id in enumerate(ids[:top_k], 1):
            row = corpus_df.iloc[doc_id]
            snippet = row["text"][:120].replace("\n", " ")
            print(f"  {rank}. [{row['category']}] {snippet}...")
        print()

# Try your own query here
search_all("interest rates and central bank policy")


In [ ]:
# Try another query
search_all("Champions League football results")